In [5]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import hnsw_benchmark as bm
import farhest_point_order as fp
from datasets import load_dataset
import os
from time import time

In [15]:
# Alapadatok megadása
# dataset_name = "arguana"
# dataset_name = "scidocs"                              # elvetve egyenlőre
dataset_name = "nfcorpus"

# embedding_model_group = "BAAI"
# embedding_model = "bge-base-en"
# embedding_model = "bge-small-en"                      # megvan LID
# embedding_model = "bge-base-en-v1.5"
# embedding_model = "bge-small-en-v1.5"

# embedding_model_group = "infgrad"
# embedding_model = "stella-base-en-v2"                 # megvan LID

embedding_model_group = "intfloat"
# embedding_model = "e5-base"
# embedding_model = "e5-small"
# embedding_model = "e5-base-v2"
embedding_model = "e5-small-v2"                       # megvan LID Farthest LPCA TwoNN
# embedding_model = "multilingual-e5-large"             # megvan LID Farthest
# embedding_model = "multilingual-e5-base"
# embedding_model = "multilingual-e5-small"

# embedding_model_group = "llmrails"
# embedding_model = "ember-v1"

# embedding_model_group = "sentence-transformers"
# embedding_model = "all-MiniLM-L6-v2"                  # megvan LID

# embedding_model_group = "TaylorAI"
# embedding_model = "bge-micro"

# embedding_model_group = "thenlper"
# embedding_model = "gte-base"

calc_lid = True
calc_fps = False

In [7]:
# Adathalmazok beolvasása
# Corpus
corpus = load_dataset("mteb/" + dataset_name, "corpus")
corpus_texts = corpus.data["corpus"]["text"]
print(f"Corpus size: {len(corpus_texts):10d}")

# Query
queries = load_dataset("mteb/" + dataset_name, "queries")
query_texts = queries.data["queries"]["text"]
print(f"Query size:  {len(query_texts):10d}")

Corpus size:       3633
Query size:        3237


In [8]:
print(corpus_texts[0])
print(query_texts[0])

Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.08–9.0 years) 6,011 participants died, of which 3,619 (60.2%) was due to breast cancer. After adjustment for age, tumor characteris

In [16]:
# Embedding végrehajtása
model = SentenceTransformer(embedding_model_group + "/" + embedding_model)

corpus_embeddings = model.encode(
    [f"passage: {text}" for text in corpus_texts],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)

query_embeddings = model.encode(
    [f"query: {text}" for text in query_texts],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)


corpus_embeddings = np.asarray(corpus_embeddings, dtype=np.float32)
query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

Batches: 100%|██████████| 102/102 [00:15<00:00,  6.68it/s]


In [17]:
print(f"Corpus dimensions: {corpus_embeddings.shape}")
print(f"Query dimensions:  {query_embeddings.shape}")

Corpus dimensions: (3633, 384)
Query dimensions:  (3237, 384)


In [18]:
# LID számolás
if calc_lid:
    corpus_lids = bm.calculate_lid(corpus_embeddings)

In [19]:
# Farthest points igazi számítása
if calc_fps:
    corpus_farthest_scores = fp.farthest_point_order(corpus_embeddings)

In [20]:
# Fájlok kiírása
# Corpus
df_corpus = pd.DataFrame(columns=["vectors"], data=[[v] for v in corpus_embeddings])
corpus_filename = dataset_name + "_corpus_" + embedding_model

if calc_lid:
    df_corpus = df_corpus.assign(LID=corpus_lids)
    corpus_filename = corpus_filename + "_lid"
if calc_fps:
    df_corpus = df_corpus.assign(fp_score=corpus_farthest_scores)
    corpus_filename = corpus_filename + "_fps"

corpus_output_path = os.path.join("..\\data\\", corpus_filename + ".json")
df_corpus.to_json(corpus_output_path, orient="records", lines=True)


# Query
df_query = pd.DataFrame(columns=["vectors"], data=[[v] for v in query_embeddings])
query_filename = dataset_name + "_query_" + embedding_model
query_output_path = os.path.join("..\\data\\", query_filename + ".json")
df_query.to_json(query_output_path, orient="records", lines=True)

In [ ]:
print(df_corpus.keys())
print(len(corpus_lids))

Index(['vectors', 'LID'], dtype='str')
3633


In [14]:
print(np.mean(corpus_lids))

19.731508528992652


In [21]:
# KNN számítás minden pontra
k = 100
start = time()
data = np.asarray([np.asarray(v) for v in corpus_embeddings], dtype=np.float32)
print(f"Init:         {((time() - start) * 1000):.2f} ms")
start = time()
scores = data @ data.T
print(f"Inversion:    {((time() - start) * 1000):.2f} ms")

# Assign the same ID to identical vectors, then exclude them from each row.
start = time()
_, vector_ids = np.unique(data, axis=0, return_inverse=True)
for row, vector_id in enumerate(vector_ids):
    scores[row, vector_ids == vector_id] = -np.inf
print(f"Filter 0:     {((time() - start) * 1000):.2f} ms")

start = time()
topk = np.argpartition(
    -scores,
    kth=k-1,
    axis=1,
)[:, :k]
print(f"TopK:         {((time() - start) * 1000):.2f} ms")

# Optional: sort those k results by actual score
start = time()
rows = np.arange(len(data))[:, None]
order = np.argsort(-scores[rows, topk], axis=1)

knn_matrix = np.take_along_axis(topk, order, axis=1)
print(f"Sort:         {((time() - start) * 1000):.2f} ms")

# LID számítása egyes vektorokra
lid_values = np.full(len(data), np.nan, dtype=np.float64)

sumdist = 0
sumcalc = 0
start = time()
for i, neighbor_indices in enumerate(knn_matrix):
    #Távolságok számítása
    sd = time()
    distances = np.array([bm.euclidian_distance(data[i], data[j]) for j in neighbor_indices])

    distances = distances[distances > 0]

    if len(distances) < k:
        print(f"Point: {i}")
        print(knn_matrix[i][:10])
        print(neighbor_indices[:10])
        raise ValueError(f"Some elements deleted.")
    sumdist += time() - sd

    # LID érték kiszámítása
    sc = time()
    r_k = distances[-1]
    sum_log = np.sum(np.log2(distances / r_k)) # ln lenne helyes, de mindegy a rangsorolás szempontjából, mivel mindegyik log fv. szig. mon. növ.
    if sum_log == 0 or k == 0:
        print(i)
        print(neighbor_indices[:10])
        exit
    lid_values[i] = -((sum_log / k) ** -1)
    sumcalc = time() - sc

print(f"LID all:      {((time() - start) * 1000):.2f} ms")
print(f"  LID calc:   {(sumcalc * 1000):.2f} ms")
print(f"  dist calc:  {(sumdist * 1000):.2f} ms")

Init:         31.74 ms
Inversion:    80.40 ms
Filter 0:     77.94 ms
TopK:         104.77 ms
Sort:         34.14 ms
LID all:      60639.42 ms
  LID calc:   0.04 ms
  dist calc:  60489.45 ms


In [26]:
i = 1056
neighbor_indices = knn_matrix[i]
distances = np.array([bm.euclidian_distance(data[i], data[j]) for j in neighbor_indices])
print(distances[:10])
# knn_scores = [float(1 - scores[i][j]) for j in neighbor_indices]
knn_scores = np.take_along_axis(-scores, order, axis=1)[i]
print(knn_scores[:10])
# print(topk[i])

[0.51192844 0.5219196  0.526099   0.53017366 0.5355582  0.53681207
 0.54022646 0.54456615 0.5448431  0.54850185]
[-0.807337   -0.81763846 -0.77339625 -0.79471254 -0.7929521  -0.79185057
 -0.798269   -0.7944218  -0.76404    -0.7839405 ]


In [ ]:
print(bm.euclidian_distance(data[i],data[neighbor_indices[0]]))
print(bm.euclidian_distance(data[i],data[neighbor_indices[1]]))
print(bm.euclidian_distance(data[neighbor_indices[0]],data[neighbor_indices[1]]))

0.41191527
0.41191527
0.0


In [ ]:
print(data[neighbor_indices[2]][:10])
print(data[i][:10])
print(scores[i][1])

print(np.sqrt(np.sum((data[i] - data[neighbor_indices[1]]) ** 2)))

[ 0.00110431  0.12936072  0.06068776  0.0475396   0.0003884   0.04552895
  0.07236189 -0.05235871 -0.02698787 -0.01762399]
[-0.0050266   0.12544452  0.06386289  0.03028075 -0.00415824  0.0580871
  0.03144125 -0.04005827 -0.00031896  0.00974478]
-0.11654803
0.41191527
